In [10]:
# YouTube comments — data preprocessing
from pathlib import Path
import html
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

DATA_PATH = Path('data/youtube_comments.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/youtube_comments.csv')
REQUIRED_COLUMNS = ['VideoTitle', 'CommentText']

In [11]:
df = pd.read_csv(DATA_PATH, usecols=REQUIRED_COLUMNS)
df.head()

,VideoTitle,CommentText
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”


## Inspect the raw dataset

In [12]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', int(df.duplicated().sum()))
df['raw_comment_length'] = df['CommentText'].fillna('').astype(str).str.len()
df['raw_token_count'] = df['CommentText'].fillna('').astype(str).str.split().str.len()
display(df[['raw_comment_length', 'raw_token_count']].describe().round(2))

Shape: (17657, 2)
Columns: ['VideoTitle', 'CommentText']


,missing
VideoTitle,0
CommentText,6


Duplicate rows: 695


,raw_comment_length,raw_token_count
count,17657.00,17657.00
mean,79.16,14.31
std,174.44,28.83
min,0.00,0.00
25%,23.00,4.00
50%,44.00,8.00
75%,82.00,16.00
max,7094.00,1573.00


## Remove non-English comments

In [13]:
def is_english_comment(text: str) -> bool:
    text = html.unescape(str(text))
    letters = [char for char in text if char.isalpha()]
    if len(letters) < 2:
        return False
    latin_ratio = sum(char.lower() in 'abcdefghijklmnopqrstuvwxyz' for char in letters) / len(letters)
    if latin_ratio < 0.85:
        return False
    words = re.findall(r'[A-Za-z]+', text.lower())
    common_english_words = sum(word in ENGLISH_STOP_WORDS for word in words)
    return latin_ratio >= 0.95 or common_english_words > 0

df['is_english'] = df['CommentText'].fillna('').map(is_english_comment)
print(df['is_english'].value_counts())
df = df[df['is_english']].copy()
df = df.drop(columns='is_english')
print('Rows after English-language filtering:', len(df))

is_english
True     16174
False     1483
Name: count, dtype: int64
Rows after English-language filtering: 16174


## Normalize, clean, and tokenize comments

In [14]:
URL_RE = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r'\s+')
WORD_RE = re.compile(r'\b\w+\b', flags=re.UNICODE)

def clean_and_tokenize(text: str) -> tuple[str, list[str]]:
    text = html.unescape(str(text))
    text = unicodedata.normalize('NFKC', text).lower()
    text = URL_RE.sub(' ', text)
    text = text.replace('_', ' ')
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    text = WHITESPACE_RE.sub(' ', text).strip()
    tokens = [token for token in WORD_RE.findall(text) if token not in ENGLISH_STOP_WORDS and len(token) > 1]
    return ' '.join(tokens), tokens

cleaned = df['CommentText'].map(clean_and_tokenize)
df['clean_comment'] = cleaned.str[0]
df['comment_tokens'] = cleaned.str[1]
df = df[df['clean_comment'].str.len().gt(0)].copy()
df = df.drop_duplicates(subset=['VideoTitle', 'CommentText']).reset_index(drop=True)
df[['CommentText', 'clean_comment', 'comment_tokens']].head()

,CommentText,clean_comment,comment_tokens
0,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim 60 youtube shorts,"[claim, 60, youtube, shorts]"
1,I will try these too 😊,try,[try]
2,Career ladder,career ladder,"[career, ladder]"
3,Very helpful,helpful,[helpful]
4,Im not proud of doing “brainrot”,im proud doing brainrot,"[im, proud, doing, brainrot]"


## Convert comments to TF-IDF features

In [15]:
tfidf = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_tfidf = tfidf.fit_transform(df['clean_comment'])
print('Rows:', X_tfidf.shape[0])
print('Features:', X_tfidf.shape[1])

Rows: 15440
Features: 14869


## Preprocessing output

In [16]:
df[['VideoTitle', 'CommentText', 'clean_comment', 'comment_tokens']].head()
X_tfidf.shape

(15440, 14869)